# Deep SVDD with the packet extractor in the PL

Everything after reading the capture runs in the FPGA:

```
PS (ARM)                         PL (FPGA, 100 MHz)
pcap file / eth0 bytes ──DMA──►  pcap parser → flow table → 8 features
                                 → preprocessing (log1p, clip, Q24) → Deep SVDD
flow records + scores  ◄──DMA──  one 128-byte record per finished flow
```

The ARM only copies the capture in 1 MB pieces. Run the cells in order. Jupyter on PYNQ runs as root, which the hardware access needs.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('..'))      # the pynq-pl folder

import numpy as np
from svdd_pl.driver import PLEngine
from svdd_pl.stream import pcap_chunks, DEFAULT_SETTINGS
from svdd_pl.records import split, flow_rows, CSV_FIELDS

eng = PLEngine(board='auto')       # board='pynq-z2' or 'zedboard' if it can't tell
print(eng.info)

## 1. A recorded capture through the PL

`deploy.sh` copies `samples/unsw_sample.pcap`, the first 100k packets of UNSW-NB15 `1.pcap`.

In [ ]:
PCAP = os.path.abspath('../samples/unsw_sample.pcap')
rows, markers, t0 = [], [], time.perf_counter()
for chunk in pcap_chunks([PCAP]):
    flows, mk = split(eng.process(chunk))
    rows += flow_rows(flows)
    markers += mk
dt = time.perf_counter() - t0
m = markers[-1]
print(f"{m['frames']:,} packets -> {len(rows):,} flows in {dt:.2f}s, all in the PL")
print(f"{m['ipv4']:,} IPv4, {m['non_ipv4']:,} other, {m['malformed']:,} malformed; peak {m['peak_flows']:,} open flows")
a = CSV_FIELDS.index('ANOMALY')
print(f"anomalies: {sum(int(r[a]) for r in rows):,}")

## 2. Same capture through the software pipeline (svdd_ps) on the ARM

Every flow, feature, input code and score must be identical. This takes longer: it is the Python extractor from pynq-ps.

In [ ]:
from svdd_pl.reference import reference_rows, compare, print_compare
t0 = time.perf_counter()
ref, _ = reference_rows([PCAP], DEFAULT_SETTINGS)
print(f"svdd_ps on the ARM: {time.perf_counter() - t0:.1f}s")
print_compare(compare(rows, ref), sys.stdout);

## 3. Highest-scoring flows

In [ ]:
s = CSV_FIELDS.index('SCORE')
for r in sorted(rows, key=lambda r: -float(r[s]))[:10]:
    print(f"{float(r[s]):10.4f}  {r[2]}:{r[3]} -> {r[4]}:{r[5]} proto {r[6]}  ({r[CSV_FIELDS.index('END_REASON')]})")

## 4. (Optional) Live traffic on eth0

Frames are packed into pcap records on the ARM and sent to the PL every 0.5 s. The model was trained on UNSW-NB15 traffic, so expect many alerts on an ordinary network.

In [ ]:
from svdd_ps.pcap import LiveCapture
from svdd_pl.stream import LiveChunker
cap, chunker = LiveCapture('eth0'), LiveChunker()
deadline, next_cut, n = time.monotonic() + 30, time.monotonic() + 0.5, 0
try:
    for ts, _, frame in cap:
        if frame is not None:
            chunker.add(ts, frame)
        if time.monotonic() >= next_cut:
            flows, _ = split(eng.process(chunker.chunk(int(time.time() * 1e6))))
            for r in flow_rows(flows):
                n += 1
                if int(r[a]):
                    print(f"ANOMALY {r[s]} {r[2]}:{r[3]} -> {r[4]}:{r[5]} proto {r[6]}")
            next_cut = time.monotonic() + 0.5
        if time.monotonic() > deadline:
            break
finally:
    cap.close()
flows, _ = split(eng.process(chunker.chunk(int(time.time() * 1e6), flush=True)))
print(f"{n + len(flows)} flows in 30 s")